In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Criar um plugin de transpilador

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou mais recentes.

```
qiskit[all]~=2.3.0
```
</details>
Criar um [plugin de transpilador](transpiler-plugins) é uma ótima forma de compartilhar seu código de transpilação com a comunidade Qiskit, permitindo que outros usuários se beneficiem da funcionalidade que você desenvolveu. Obrigado pelo seu interesse em contribuir com a comunidade Qiskit!

Antes de criar um plugin de transpilador, você precisa decidir qual tipo de plugin é adequado para sua situação. Existem três tipos de plugins de transpilador:
- [**Plugin de estágio do transpilador**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins). Escolha este se você estiver definindo um gerenciador de passes que pode substituir um dos [6 estágios](transpiler-stages) de um gerenciador de passes staged predefinido.
- [**Plugin de síntese unitária**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin). Escolha este se o seu código de transpilação recebe como entrada uma matriz unitária (representada como um array Numpy) e produz como saída uma descrição de um Circuit quântico que implementa esse unitário.
- [**Plugin de síntese de alto nível**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin). Escolha este se o seu código de transpilação recebe como entrada um "objeto de alto nível", como um operador Clifford ou uma função linear, e produz como saída uma descrição de um Circuit quântico que implementa esse objeto de alto nível. Objetos de alto nível são representados por subclasses da classe [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

Depois de determinar qual tipo de plugin criar, siga estes passos para criar o plugin:

1. Crie uma subclasse da classe abstrata de plugin adequada:
   - [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin) para um plugin de estágio do transpilador,
   - [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) para um plugin de síntese unitária, e
   - [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) para um plugin de síntese de alto nível.
2. Exponha a classe como um [entry point do setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) nos metadados do pacote, geralmente editando o arquivo `pyproject.toml`, `setup.cfg` ou `setup.py` do seu pacote Python.

Não há limite para o número de plugins que um único pacote pode definir, mas cada plugin deve ter um nome único. O próprio SDK do Qiskit inclui uma série de plugins, cujos nomes também são reservados. Os nomes reservados são:

- Plugins de estágio do transpilador: Veja [esta tabela](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages).
- Plugins de síntese unitária: `default`, `aqc`, `sk`
- Plugins de síntese de alto nível:

| Classe de operação | Nome da operação | Nomes reservados |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`, `ag`, `bm`, `greedy`, `layers`, `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`, `kms`, `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`, `kms`, `basic`, `acg`, `token_swapper` |

Nas próximas seções, mostramos exemplos desses passos para os diferentes tipos de plugins. Nestes exemplos, assumimos que estamos criando um pacote Python chamado `my_qiskit_plugin`. Para informações sobre como criar pacotes Python, você pode conferir [este tutorial](https://packaging.python.org/en/latest/tutorials/packaging-projects/) no site do Python.
## Exemplo: Criar um plugin de estágio do transpilador
Neste exemplo, criamos um plugin de estágio do transpilador para o estágio `layout` (veja [Estágios do transpilador](transpiler-stages) para uma descrição dos 6 estágios do pipeline de transpilação integrado do Qiskit).
Nosso plugin simplesmente executa o [VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout) por um número de tentativas que depende do nível de otimização solicitado.

Primeiro, criamos uma subclasse de [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin). Há um método que precisamos implementar, chamado [`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager). Esse método recebe como entrada um [PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig) e retorna o gerenciador de passes que estamos definindo. O objeto PassManagerConfig armazena informações sobre o backend alvo, como seu mapa de acoplamento e gates de base.

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

Agora, expomos o plugin adicionando um entry point nos metadados do nosso pacote Python.
Aqui, assumimos que a classe que definimos está exposta em um módulo chamado `my_qiskit_plugin`, por exemplo, sendo importada no arquivo `__init__.py` do módulo `my_qiskit_plugin`.
Editamos o arquivo `pyproject.toml`, `setup.cfg` ou `setup.py` do nosso pacote (dependendo de qual tipo de arquivo você escolheu para armazenar os metadados do seu projeto Python):

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>
Veja a [tabela de estágios de plugins do transpilador](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table) para os entry points e expectativas de cada estágio do transpilador.

Para verificar se o seu plugin é detectado com sucesso pelo Qiskit, instale o seu pacote de plugin e siga as instruções em [Plugins do transpilador](transpiler-plugins#list-available-transpiler-stage-plugins) para listar os plugins instalados, e certifique-se de que o seu plugin aparece na lista:

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

Se o nosso plugin de exemplo estivesse instalado, o nome `my_layout` apareceria nessa lista.

Se você quiser usar um estágio de transpilador integrado como ponto de partida para o seu plugin de estágio do transpilador, você pode obter o gerenciador de passes de um estágio de transpilador integrado usando [PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). O seguinte bloco de código mostra como fazer isso para obter o estágio de otimização integrado para o nível de otimização 3.

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## Exemplo: Criar um plugin de síntese unitária
Neste exemplo, criaremos um plugin de síntese unitária que simplesmente usa o passe de transpilação integrado [UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) para sintetizar um gate. Claro, o seu próprio plugin fará algo mais interessante do que isso.

A classe [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) define a interface e o contrato para plugins de síntese unitária. O método principal é
[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run),
que recebe como entrada um array Numpy armazenando uma matriz unitária
e retorna um [DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit) representando o Circuit sintetizado a partir dessa matriz unitária.
Além do método `run`, há uma série de métodos de propriedade que precisam ser definidos.
Veja [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) para documentação de todas as propriedades obrigatórias.

Vamos criar nossa subclasse de UnitarySynthesisPlugin:

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

Se você achar que as entradas disponíveis para o método [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)
são insuficientes para seus propósitos, [abra uma issue](https://github.com/Qiskit/qiskit/issues/new/choose) explicando seus requisitos. Alterações na interface do plugin, como adicionar entradas opcionais adicionais, serão feitas de forma compatível com versões anteriores, de modo que não exijam mudanças nos plugins existentes.

> **Note:** Todos os métodos prefixados com `supports_` são reservados em uma classe derivada de `UnitarySynthesisPlugin` como parte da interface. Você não deve definir nenhum método `supports_*` personalizado em uma subclasse que não esteja definido na classe abstrata.

Agora, expomos o plugin adicionando um entry point nos metadados do nosso pacote Python.
Aqui, assumimos que a classe que definimos está exposta em um módulo chamado `my_qiskit_plugin`, por exemplo, sendo importada no arquivo `__init__.py` do módulo `my_qiskit_plugin`.
Editamos o arquivo `pyproject.toml`, `setup.cfg` ou `setup.py` do nosso pacote:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

Como antes, se o seu projeto usar `setup.cfg` ou `setup.py` em vez de `pyproject.toml`, consulte a [documentação do setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) para saber como adaptar essas linhas à sua situação.

Para verificar se o seu plugin é detectado com sucesso pelo Qiskit, instale o seu pacote de plugin e siga as instruções em [Plugins do transpilador](transpiler-plugins#list-available-unitary-synthesis-plugins) para listar os plugins instalados, e certifique-se de que o seu plugin aparece na lista:

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

Se o nosso plugin de exemplo estivesse instalado, o nome `my_unitary_synthesis` apareceria nessa lista.

Para acomodar plugins de síntese unitária que expõem múltiplas opções,
a interface do plugin tem uma opção para que os usuários forneçam um
dicionário de configuração de forma livre. Isso será passado ao método `run`
via o argumento de palavra-chave `options`. Se o seu plugin tiver essas opções de configuração, você deve documentá-las claramente.

## Exemplo: Criar um plugin de síntese de alto nível

Neste exemplo, criaremos um plugin de síntese de alto nível que simplesmente usa a função integrada [synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm) para sintetizar um operador Clifford.

A classe [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) define a interface e o contrato para plugins de síntese de alto nível. O método principal é [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run).
O argumento posicional `high_level_object` é uma [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) que representa o objeto de "alto nível" a ser sintetizado. Por exemplo, pode ser uma
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) ou um
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford).
Os seguintes argumentos de palavra-chave estão presentes:
- `target` especifica o backend alvo, permitindo ao plugin
acessar todas as informações específicas do alvo,
como o mapa de acoplamento, o conjunto de gates suportados e assim por diante
- `coupling_map` especifica apenas o mapa de acoplamento e é usado somente quando `target` não é especificado.
- `qubits` especifica a lista de Qubits sobre os quais o
objeto de alto nível é definido, caso a síntese seja feita no Circuit físico.
Um valor ``None`` indica que o layout ainda não foi escolhido e os Qubits físicos no alvo ou mapa de acoplamento em que esta operação está atuando ainda não foram determinados.
- `options`, um dicionário de configuração de forma livre para opções específicas do plugin. Se o seu plugin tiver essas opções de configuração, você
deve documentá-las claramente.

O método `run` retorna um [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)
representando o Circuit sintetizado a partir desse objeto de alto nível.
Também é permitido retornar `None`, indicando que o plugin não consegue sintetizar o objeto de alto nível fornecido.
A síntese real de objetos de alto nível é realizada pelo passe de transpilador
[HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis).

Além do método `run`, há uma série de métodos de propriedade que precisam ser definidos.
Veja [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) para documentação de todas as propriedades obrigatórias.

Vamos definir nossa subclasse de HighLevelSynthesisPlugin: